# Librerías esenciales para IA con Python

NumPy, Pandas, Matplotlib, Requests/HTTPX y utilidades que aparecen en todos los proyectos de IA.

In [ ]:
# pip install numpy pandas matplotlib seaborn httpx python-dotenv tqdm
import numpy as np
import pandas as pd
import json, os
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)

## 1. NumPy — operaciones sobre embeddings

In [ ]:
import numpy as np

# Simular embeddings (en producción vienen de la API)
np.random.seed(42)
emb_a = np.random.randn(1536)  # dimensión OpenAI text-embedding-3-small
emb_b = np.random.randn(1536)
emb_similar = emb_a + np.random.randn(1536) * 0.1  # muy parecido a emb_a

def similitud_coseno(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f'Similitud A-B (aleatorios):    {similitud_coseno(emb_a, emb_b):.4f}')
print(f'Similitud A-Similar (cercano): {similitud_coseno(emb_a, emb_similar):.4f}')

# Operaciones útiles con arrays de embeddings
base = np.random.randn(100, 1536)  # 100 documentos
query = np.random.randn(1536)

# Búsqueda del más similar (vectorizada, sin bucle)
normas = np.linalg.norm(base, axis=1, keepdims=True)
base_norm = base / normas
query_norm = query / np.linalg.norm(query)
scores = base_norm @ query_norm
top5 = np.argsort(scores)[-5:][::-1]
print(f'\nTop-5 documentos más similares: {top5}')
print(f'Sus scores: {scores[top5].round(4)}')

## 2. Pandas — gestión de datasets de IA

In [ ]:
import pandas as pd

# Dataset de ejemplo: evaluaciones de LLM
datos = {
    'pregunta': [
        '¿Cuál es la capital de Francia?',
        '¿Qué es un transformer?',
        '¿Cómo funciona RAG?',
        '¿Qué es fine-tuning?',
        '¿Diferencia entre GPT y BERT?',
    ],
    'respuesta_modelo': [
        'París',
        'Es una arquitectura de red neuronal basada en atención.',
        'Retrieval-Augmented Generation combina búsqueda con generación.',
        'Ajuste fino de un modelo preentrenado.',
        'GPT es generativo, BERT es bidireccional para comprensión.',
    ],
    'puntuacion': [1.0, 0.9, 0.85, 0.8, 0.75],
    'tokens_entrada': [15, 12, 14, 13, 18],
    'tokens_salida': [3, 18, 22, 15, 20],
    'modelo': ['haiku', 'haiku', 'sonnet', 'haiku', 'sonnet'],
    'latencia_ms': [320, 480, 1200, 390, 1100],
}

df = pd.DataFrame(datos)

# Análisis básico
print('=== Resumen por modelo ===')
print(df.groupby('modelo').agg(
    n_llamadas=('puntuacion', 'count'),
    puntuacion_media=('puntuacion', 'mean'),
    latencia_media_ms=('latencia_ms', 'mean'),
    tokens_salida_total=('tokens_salida', 'sum'),
).round(2))

# Calcular coste
precios = {'haiku': (0.80, 4.00), 'sonnet': (3.00, 15.00)}
df['coste_usd'] = df.apply(
    lambda r: (
        r['tokens_entrada'] * precios[r['modelo']][0] +
        r['tokens_salida'] * precios[r['modelo']][1]
    ) / 1_000_000,
    axis=1
)

print(f'\nCoste total: ${df["coste_usd"].sum():.6f}')
print(df[['pregunta', 'modelo', 'puntuacion', 'coste_usd']].to_string(index=False))

In [ ]:
# Pandas para construir datasets de fine-tuning
ejemplos_raw = [
    {'input': 'Clasifica: Me encantó el producto', 'label': 'positivo'},
    {'input': 'Clasifica: Llegó roto', 'label': 'negativo'},
    {'input': 'Clasifica: Está bien', 'label': 'neutro'},
    {'input': 'Clasifica: Excelente calidad', 'label': 'positivo'},
    {'input': 'Clasifica: No lo recomiendo', 'label': 'negativo'},
]

df_dataset = pd.DataFrame(ejemplos_raw)

# Convertir a formato de mensajes para fine-tuning
def a_formato_chat(row: pd.Series) -> dict:
    return {
        'messages': [
            {'role': 'user', 'content': row['input']},
            {'role': 'assistant', 'content': row['label']},
        ]
    }

dataset_chat = df_dataset.apply(a_formato_chat, axis=1).tolist()

# Guardar como JSONL (formato estándar para fine-tuning)
with open('/tmp/dataset_finetune.jsonl', 'w') as f:
    for ejemplo in dataset_chat:
        f.write(json.dumps(ejemplo, ensure_ascii=False) + '\n')

print(f'Dataset guardado: {len(dataset_chat)} ejemplos')
print('Distribución:', df_dataset['label'].value_counts().to_dict())

## 3. Matplotlib — visualizar métricas de IA

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Gráfico 1: puntuación por modelo
colores = {'haiku': '#4CAF50', 'sonnet': '#2196F3'}
for modelo, grupo in df.groupby('modelo'):
    axes[0].bar(grupo.index, grupo['puntuacion'], color=colores[modelo], label=modelo, alpha=0.8)
axes[0].set_title('Puntuación por pregunta')
axes[0].set_ylabel('Puntuación')
axes[0].legend()
axes[0].set_ylim(0, 1.1)

# Gráfico 2: latencia vs puntuación
for modelo, grupo in df.groupby('modelo'):
    axes[1].scatter(grupo['latencia_ms'], grupo['puntuacion'],
                    c=colores[modelo], label=modelo, s=100, alpha=0.8)
axes[1].set_title('Latencia vs Puntuación')
axes[1].set_xlabel('Latencia (ms)')
axes[1].set_ylabel('Puntuación')
axes[1].legend()

# Gráfico 3: coste acumulado
df_sorted = df.sort_values('coste_usd', ascending=False)
axes[2].barh(range(len(df_sorted)), df_sorted['coste_usd'] * 1000,
             color=[colores[m] for m in df_sorted['modelo']], alpha=0.8)
axes[2].set_yticks(range(len(df_sorted)))
axes[2].set_yticklabels([p[:20] + '...' for p in df_sorted['pregunta']], fontsize=8)
axes[2].set_title('Coste por llamada (milésimas $)')
axes[2].set_xlabel('Coste (mUSD)')

plt.tight_layout()
plt.savefig('/tmp/metricas_llm.png', dpi=100, bbox_inches='tight')
plt.show()
print('Gráfico guardado en /tmp/metricas_llm.png')

## 4. HTTPX — cliente HTTP moderno para APIs

In [ ]:
import httpx
import asyncio

# Cliente síncrono con retry y timeout
def crear_cliente_api(base_url: str, api_key: str, timeout: float = 30.0) -> httpx.Client:
    return httpx.Client(
        base_url=base_url,
        headers={
            'x-api-key': api_key,
            'anthropic-version': '2023-06-01',
            'content-type': 'application/json',
        },
        timeout=httpx.Timeout(timeout, connect=5.0),
    )

# Cliente async para llamadas concurrentes
async def llamar_multiples_apis(urls: list[str]) -> list[dict]:
    async with httpx.AsyncClient(timeout=10.0) as cliente:
        tareas = [cliente.get(url) for url in urls]
        respuestas = await asyncio.gather(*tareas, return_exceptions=True)
        resultados = []
        for url, resp in zip(urls, respuestas):
            if isinstance(resp, Exception):
                resultados.append({'url': url, 'error': str(resp)})
            else:
                resultados.append({'url': url, 'status': resp.status_code})
        return resultados

# Test con API pública
urls_test = [
    'https://httpbin.org/status/200',
    'https://httpbin.org/status/404',
    'https://httpbin.org/delay/1',
]

# Solo ejecutar si hay conexión
try:
    resultados = await llamar_multiples_apis(urls_test)
    for r in resultados:
        print(r)
except Exception as e:
    print(f'Sin conexión: {e}')
    print('Ejemplo de uso del cliente:', crear_cliente_api('https://api.anthropic.com', 'key'))

## 5. tqdm — barras de progreso para procesamiento por lotes

In [ ]:
from tqdm.notebook import tqdm
import time

documentos = [f'Documento {i}: texto de prueba para procesar con IA.' for i in range(20)]

def procesar_documento(texto: str) -> dict:
    time.sleep(0.05)  # simula llamada a API
    return {'longitud': len(texto), 'palabras': len(texto.split())}

# Sin barra de progreso — no sabes cuánto falta
# resultados = [procesar_documento(d) for d in documentos]

# Con tqdm — feedback visual
resultados = []
for doc in tqdm(documentos, desc='Procesando documentos'):
    resultado = procesar_documento(doc)
    resultados.append(resultado)

total_palabras = sum(r['palabras'] for r in resultados)
print(f'\nProcesados: {len(resultados)} documentos, {total_palabras} palabras en total')

# tqdm con asyncio y APIs reales
from tqdm.asyncio import tqdm_asyncio

async def procesar_async(doc: str) -> dict:
    await asyncio.sleep(0.05)
    return {'longitud': len(doc)}

resultados_async = await tqdm_asyncio.gather(
    *[procesar_async(d) for d in documentos[:5]],
    desc='Procesando async'
)
print(f'Async: {len(resultados_async)} resultados')